In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')

In [2]:
import os
import json
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)

load_dotenv()
kaggle_username = os.getenv('KAGGLE-USERNAME')
kaggle_key = os.getenv('KAGGLE-KEY')

## Load Data

In [3]:
# buat folder
os.makedirs('../data', exist_ok=True)

# download data dari kaggle
!kaggle datasets download -d shahriarkabir/us-logistics-performance-dataset --unzip -p ../data

# cek isi folder
print(os.listdir('../data'))

Dataset URL: https://www.kaggle.com/datasets/shahriarkabir/us-logistics-performance-dataset
License(s): CC0-1.0
100%|█████████████████████████████████████| 41.4k/41.4k [00:00<00:00, 1.43MB/s]

['.gitkeep', 'logistics_shipments_dataset.csv']


In [5]:
df = pd.read_csv('../data/logistics_shipments_dataset.csv')
print(df.shape)
print(df.columns.tolist())   # verifikasi nama kolom
df.head()

(2000, 11)
['Shipment_ID', 'Origin_Warehouse', 'Destination', 'Carrier', 'Shipment_Date', 'Delivery_Date', 'Weight_kg', 'Cost', 'Status', 'Distance_miles', 'Transit_Days']


,Shipment_ID,Origin_Warehouse,Destination,Carrier,Shipment_Date,Delivery_Date,Weight_kg,Cost,Status,Distance_miles,Transit_Days
0,SH10000,Warehouse_MIA,San Francisco,UPS,2023-10-02,2023-10-04,25.7,67.46,Delivered,291,2
1,SH10001,Warehouse_MIA,Atlanta,DHL,2023-12-06,2023-12-09,38.9,268.85,Delivered,1225,3
2,SH10002,Warehouse_LA,Houston,DHL,2023-09-18,2023-09-20,37.2,74.35,Delivered,220,2
3,SH10003,Warehouse_BOS,Seattle,OnTrac,2023-01-26,2023-02-04,42.6,187.04,Delivered,1156,9
4,SH10004,Warehouse_SF,Dallas,OnTrac,2023-06-03,2023-06-06,7.9,120.01,Delivered,1017,3


In [ ]:
df.info()
df.describe(include='all').T

## Processing Data

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplikat:', df.duplicated().sum())
print('\nDistribusi Status:')
print(df['Status'].value_counts())

In [ ]:
df = df.drop_duplicates()
df = df[df['Status'].isin(['Delivered', 'Delayed'])].copy()
df['target'] = (df['Status'] == 'Delayed').astype(int)
print(df['target'].value_counts(normalize=True).round(3))

In [ ]:
df['Shipment_Date'] = pd.to_datetime(df['Shipment_Date'], errors='coerce')
d = df['Shipment_Date'].dt
df['ship_month'] = d.month
df['ship_dayofweek'] = d.dayofweek
df['ship_is_weekend'] = (d.dayofweek >= 5).astype(int)
df['ship_quarter'] = d.quarter

In [ ]:
# Fitur 
NUM_FEATURES = ['Weight_kg', 'Cost', 'Distance_miles',
                'ship_month', 'ship_dayofweek', 'ship_is_weekend',
                'ship_quarter']
CAT_FEATURES = ['Origin_Warehouse', 'Destination', 'Carrier']

X = df[NUM_FEATURES + CAT_FEATURES].copy()
y = df['target'].copy()
print('fitur:', X.shape[1], '| baris:', len(X))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df[['Weight_kg', 'Cost']], ax=ax[0]); ax[0].set_title('Outlier: Weight & Cost')
sns.barplot(x='Carrier', y='target', data=df, ax=ax[1]); ax[1].set_title('Rasio Delay per Carrier')
ax[1].tick_params(axis='x', rotation=45); plt.tight_layout()

In [ ]:
# Hitung outlier numerik via IQR
for col in ['Weight_kg', 'Cost', 'Distance_miles']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f'{col:15s}: {n_out} outlier')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', RobustScaler())]), NUM_FEATURES),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_FEATURES),
])
print(X_train.shape, X_test.shape)

In [ ]:
results = {}

def evaluate(name, model, X_te, y_te, proba=None):
    pred = model.predict(X_te) if proba is None else (proba >= 0.5).astype(int)
    p = proba if proba is not None else model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)
    f1 = f1_score(y_te, pred)
    results[name] = {'F1': f1, 'ROC_AUC': auc}
    print(f'=== {name} ===')
    print(classification_report(y_te, pred, digits=3))
    print(f'ROC-AUC: {auc:.3f}\n')

## Modeling

In [ ]:
logreg = Pipeline([('prep', preprocessor),
                   ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
logreg.fit(X_train, y_train)
evaluate('LogisticRegression', logreg, X_test, y_test)

In [ ]:
rf = Pipeline([('prep', preprocessor),
               ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                n_jobs=-1, random_state=RANDOM_STATE))])
rf.fit(X_train, y_train)
evaluate('RandomForest', rf, X_test, y_test)

In [ ]:
from catboost import CatBoostClassifier

# scale_pos_weight = rasio kelas negatif/positif untuk imbalance
spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
cb = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                        scale_pos_weight=spw, random_seed=RANDOM_STATE, verbose=0)
cb.fit(X_train, y_train, cat_features=CAT_FEATURES)
evaluate('CatBoost', cb, X_test, y_test)

In [ ]:
# Validasi silang (stratified) untuk cek konsistensi
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
print(f'CV F1: {scores.mean():.3f} +/- {scores.std():.3f}')

In [ ]:
imp = pd.Series(cb.get_feature_importance(),
                index=NUM_FEATURES + CAT_FEATURES).sort_values()
imp.plot(kind='barh', figsize=(8, 5)); plt.title('Feature Importance (CatBoost)'); plt.tight_layout()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Xtr = preprocessor.fit_transform(X_train).astype('float32')
Xte = preprocessor.transform(X_test).astype('float32')
ytr = y_train.values.astype('float32').reshape(-1, 1)

train_dl = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                      batch_size=64, shuffle=True)
pos_weight = torch.tensor([spw], dtype=torch.float32)
print('input dim:', Xtr.shape[1])

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), 
            nn.ReLU(), 
            nn.BatchNorm1d(128), 
            nn.Dropout(0.3),
            nn.Linear(128, 64), 
            nn.ReLU(), 
            nn.Dropout(0.2),
            nn.Linear(64, 1)) 
    def forward(self, x):
        return self.net(x)

model = MLPClassifier(Xtr.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

In [ ]:
for ep in range(1, 41):
    model.train(); total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(xb)
    if ep % 10 == 0 or ep == 1:
        print(f'epoch {ep:2d} | loss {total/len(train_dl.dataset):.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(Xte).to(device)).cpu().numpy().ravel()
proba = 1 / (1 + np.exp(-logits))   # sigmoid
evaluate('MLP (PyTorch)', None, X_test, y_test, proba=proba)

## Evaluasi

In [ ]:
res_df = pd.DataFrame(results).T.sort_values('F1', ascending=False)
display(res_df)
res_df.plot(kind='bar', figsize=(8, 4)); plt.title('Perbandingan Model'); plt.ylim(0, 1); plt.tight_layout()

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, cb.predict(X_test)),
                       display_labels=['On-time', 'Delayed']).plot(cmap='Blues')
plt.title('CatBoost — Confusion Matrix'); plt.tight_layout()

In [ ]:
iso = IsolationForest(contamination=0.03, random_state=RANDOM_STATE)
df['anomaly'] = iso.fit_predict(df[['Weight_kg', 'Cost', 'Distance_miles']].fillna(df.median(numeric_only=True)))
print('Anomali terdeteksi:', (df['anomaly'] == -1).sum())
df[df['anomaly'] == -1][['Shipment_ID', 'Weight_kg', 'Cost', 'Distance_miles']].head()

In [ ]:
import joblib
joblib.dump(cb, 'output/model_delay_catboost.pkl')
print('saved -> output/model_delay_catboost.pkl')